# Mat & Comp – ML 12: Proyecto final de machine learning
## De los datos crudos a un modelo evaluado

**Curso:** Machine learning (Sección 3) · **Duración sugerida:** 2 o 3 sesiones de 50 minutos · **Requisito:** ML 1 a ML 11

### Objetivos de aprendizaje
Al terminar este proyecto podrás:
1. Recorrer **todas las etapas** de un proyecto de machine learning, desde la pregunta hasta las conclusiones.
2. **Limpiar** datos reales y desordenados: duplicados, textos inconsistentes, números mal escritos, valores imposibles y faltantes.
3. Convertir columnas de **categorías** en números (codificación *one-hot*) y armar un `Pipeline` completo con `ColumnTransformer`.
4. Comparar modelos con **validación cruzada** frente a una **línea base**, ajustar hiperparámetros con `GridSearchCV` y evaluar **una sola vez** en la prueba.
5. Detectar una **fuga de información** y comunicar resultados con sus **limitaciones**.

## 🔍 ¿Quién necesitará ayuda?

Tres colegios hicieron una encuesta a sus estudiantes de grados 9 a 11 antes de una prueba de matemáticas, y después anotaron el puntaje de cada uno. Quieren saber, **antes** de la próxima prueba, qué estudiantes probablemente no aprobarán, para ofrecerles tutorías a tiempo.

✏️ *Si tú fueras el equipo de datos, ¿qué harías primero? Haz una lista de 5 pasos.*

**Mi respuesta:**

Así se ve un proyecto de machine learning de principio a fin. Cada paso usa algo que ya aprendiste:

| Etapa | Qué hacemos | Lo viste en |
|---|---|---|
| 1. La pregunta | ¿Qué queremos predecir y para qué? | — |
| 2. Cargar y revisar | Encontrar los problemas de los datos | ML 1 |
| 3. Limpiar | Arreglar o quitar lo que está mal | ML 1 |
| 4. Explorar | Gráficas y relaciones | ML 1, ML 11 |
| 5. Separar la prueba | Guardarla bajo llave | ML 7 |
| 6. Línea base y candidatos | Comparar con validación cruzada | ML 4, 5, 7, 8, 10 |
| 7. Ajustar | Elegir hiperparámetros | ML 7, ML 9 |
| 8. Evaluar una vez | Exactitud, matriz de confusión | ML 7, ML 8 |
| 9. Interpretar y comunicar | Qué aprendimos y qué no sabemos | ML 1, ML 10 |

⚠️ Los datos de este proyecto son **simulados** (inventados con fórmulas y azar) para que se parezcan a una encuesta real, incluidos sus errores. Las conclusiones sirven para aprender el método, no para decir nada sobre colegios reales.

## 1. La pregunta

**Queremos predecir** la columna `aprobo` («sí» o «no») a partir de lo que se sabe **antes** de la prueba: hábitos, colegio, grado y nota del año anterior.

**Para qué:** ofrecer tutorías. Como vimos en ML 8, dejar sin ayuda a quien la necesita es peor que ofrecerla de más, así que nos importará mucho la **sensibilidad** para detectar a quienes **no** aprueban.

## 2. Cargar y revisar: buscar los problemas

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

URL = "https://raw.githubusercontent.com/emersonjleon/compumath/main/lecciones/seccion3_ml/datos/encuesta_colegios.csv"
crudos = pd.read_csv(URL)
print(crudos.shape)
crudos.head(8)

In [ ]:
crudos.info()

✏️ *Mira la columna `Dtype`: ¿por qué `horas_estudio` aparece como `object` (texto) si deberían ser números? ¿Qué columnas tienen menos de 328 valores?*

Revisemos los valores de las columnas de texto:

In [ ]:
for col in ["colegio", "deporte", "desayuna", "aprobo"]:
    print(col, "→", crudos[col].value_counts().to_dict())

In [ ]:
crudos.describe().round(1)

Encontramos varios problemas típicos de datos reales:

1. **Textos inconsistentes:** «sí», «Sí», «SI», «si » (con espacio)… son la misma respuesta. En `colegio` aparecen «a» y «A ».
2. **Números como texto:** algunas horas de estudio se escribieron con coma decimal («3,5»), así que pandas leyó toda la columna como texto.
3. **Valores imposibles:** alguien duerme 70 horas por noche (seguramente quiso escribir 7.0), y hay estudiantes que leyeron −1 libros.
4. **Datos faltantes** en `horas_sueno` y `transporte_min`.
5. ¿**Filas duplicadas**? Revisémoslo:

In [ ]:
print("Filas duplicadas:", crudos.duplicated().sum())
print("Códigos de estudiante repetidos:", crudos["estudiante"].duplicated().sum())

## 3. Limpiar

Trabajamos sobre una **copia**, para no perder los datos originales, y arreglamos un problema a la vez, comprobando cada arreglo.

In [ ]:
datos = crudos.copy()

# 1. Duplicados
datos = datos.drop_duplicates()
print("Después de quitar duplicados:", datos.shape)

# 2. Textos: quitar espacios, pasar a minúsculas y unificar las tildes
for col in ["deporte", "desayuna", "aprobo"]:
    datos[col] = datos[col].str.strip().str.lower().str.replace("í", "i")
datos["colegio"] = datos["colegio"].str.strip().str.upper()

for col in ["colegio", "deporte", "desayuna", "aprobo"]:
    print(col, "→", datos[col].value_counts().to_dict())

In [ ]:
# 3. Números con coma decimal
datos["horas_estudio"] = datos["horas_estudio"].str.replace(",", ".").astype(float)
print("Tipo de horas_estudio:", datos["horas_estudio"].dtype)

# 4. Valores imposibles → faltantes (NaN). Mejor «no sé» que un dato falso.
print("Sueño > 14 horas:", (datos["horas_sueno"] > 14).sum(), "  libros < 0:", (datos["libros_al_año"] < 0).sum())
datos.loc[datos["horas_sueno"] > 14, "horas_sueno"] = np.nan
datos.loc[datos["libros_al_año"] < 0, "libros_al_año"] = np.nan

# 5. Columnas sí/no → 1/0
for col in ["deporte", "desayuna", "aprobo"]:
    datos[col] = (datos[col] == "si").astype(int)

print("\nFaltantes por columna:")
print(datos.isna().sum()[datos.isna().sum() > 0])

¿Por qué convertimos el 70 en faltante en lugar de cambiarlo por 7? Porque **adivinamos** que fue un error de digitación. Es razonable, pero un dato inventado puede engañar al modelo. En un proyecto real, lo ideal es preguntarle a quien recogió los datos.

Los **faltantes** todavía no los rellenamos: lo hará el `Pipeline`, usando solo los datos de entrenamiento (sección 6).

## 4. Explorar

In [ ]:
print(f"Aprobaron: {datos['aprobo'].mean():.1%} de {len(datos)} estudiantes")

fig, ejes = plt.subplots(1, 3, figsize=(14, 3.5))
for eje, col in zip(ejes, ["horas_estudio", "horas_redes", "nota_previa"]):
    for valor, color, nombre in [(1, "seagreen", "aprobó"), (0, "crimson", "no aprobó")]:
        eje.hist(datos.loc[datos["aprobo"] == valor, col].dropna(), bins=15, alpha=0.5, color=color, label=nombre)
    eje.set_title(col); eje.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(12, 3))
for eje, col in zip(ejes, ["colegio", "desayuna", "deporte"]):
    tasa = datos.groupby(col)["aprobo"].mean()
    eje.bar(tasa.index.astype(str), tasa.values, color="steelblue")
    eje.set_ylim(0, 1); eje.set_title(f"% que aprobó según {col}")
plt.tight_layout()
plt.show()

numericas = ["grado", "horas_estudio", "horas_sueno", "horas_redes", "transporte_min", "libros_al_año", "nota_previa", "aprobo"]
print(datos[numericas].corr()["aprobo"].drop("aprobo").sort_values().round(2))

✏️ *Escribe tres observaciones. ¿Qué variables parecen más relacionadas con aprobar? ¿Hay diferencias entre colegios? Recuerda (ML 1): correlación no es causalidad.*

## 5. Separar la prueba (¡y no tocarla!)

Antes de probar cualquier modelo, apartamos el 25 % de los estudiantes como **prueba**. `stratify=y` mantiene la misma proporción de aprobados en ambas partes.

⚠️ La columna `puntaje` **no** se puede usar para predecir: ¡es justamente de donde sale `aprobo`! Y el código `estudiante` no dice nada del estudiante. Las quitamos de las características.

In [ ]:
from sklearn.model_selection import train_test_split

numericas = ["grado", "horas_estudio", "horas_sueno", "horas_redes", "transporte_min", "libros_al_año", "nota_previa", "deporte", "desayuna"]
categoricas = ["colegio"]
X = datos[numericas + categoricas]
y = datos["aprobo"]

X_ent, X_pru, y_ent, y_pru = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
print("Entrenamiento:", X_ent.shape, "  Prueba:", X_pru.shape)

## 6. El pipeline: preparar los datos sin hacer trampa

### Categorías a números: *one-hot*, a mano

`colegio` es A, B o C. Si lo convirtiéramos en 1, 2, 3, el modelo creería que C es «el triple» de A, o que B está «entre» A y C. En lugar de eso se crea **una columna por categoría**, con 1 o 0:

In [ ]:
print(pd.get_dummies(X_ent["colegio"].head(6), prefix="colegio", dtype=int))

### `ColumnTransformer`: un tratamiento por tipo de columna

Para las columnas numéricas: **rellenar** faltantes con la mediana (`SimpleImputer`) y **normalizar** (`StandardScaler`, ML 4). Para las categóricas: **one-hot** (`OneHotEncoder`). `ColumnTransformer` aplica cada receta a sus columnas, y todo va dentro de un `Pipeline` para que, en la validación cruzada, la mediana y la normalización se calculen **solo** con los datos de entrenamiento de cada grupo (ML 7).

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preparar = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numericas),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categoricas),
])

ejemplo = preparar.fit_transform(X_ent)
print("Forma después de preparar:", ejemplo.shape)
print("Columnas:", list(preparar.get_feature_names_out()))

## 7. Línea base y modelos candidatos

Toda comparación necesita una **línea base**: el modelo más tonto posible. Aquí, responder siempre la clase más común (`DummyClassifier`). Si un modelo no le gana a eso, no sirve.

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

grupos = StratifiedKFold(5, shuffle=True, random_state=0)
candidatos = {
    "línea base (la más común)": DummyClassifier(strategy="most_frequent"),
    "regresión logística (ML 8)": LogisticRegression(max_iter=1000),
    "k vecinos, k = 15 (ML 4)": KNeighborsClassifier(15),
    "árbol, profundidad 4 (ML 5)": DecisionTreeClassifier(max_depth=4, random_state=0),
    "bosque aleatorio (ML 10)": RandomForestClassifier(n_estimators=300, random_state=0),
}
resultados = {}
for nombre, modelo in candidatos.items():
    tubo = Pipeline([("preparar", preparar), ("modelo", modelo)])
    puntajes = cross_val_score(tubo, X_ent, y_ent, cv=grupos)
    resultados[nombre] = puntajes
    print(f"{nombre:28}: {puntajes.mean():.1%}  (± {puntajes.std():.1%})")

In [ ]:
nombres_m = list(resultados.keys())
medias = [r.mean() for r in resultados.values()]
desv = [r.std() for r in resultados.values()]
plt.figure(figsize=(7, 3.2))
plt.barh(nombres_m, medias, xerr=desv, color="steelblue", capsize=4)
plt.xlim(0.4, 1)
plt.xlabel("exactitud con validación cruzada (barra: ± desviación entre grupos)")
plt.gca().invert_yaxis()
plt.grid(alpha=0.3)
plt.show()

✏️ *¿Qué modelo gana? ¿Es mucha la diferencia con el segundo, comparada con lo que varía entre grupos (las rayitas negras)? ¿Cuánto le ganan todos a la línea base?*

## 8. Ajustar hiperparámetros con `GridSearchCV`

Los **hiperparámetros** son las perillas que no se aprenden de los datos: el `C` de la regresión logística (ML 8 y ML 9), la `k` de k vecinos, la profundidad del árbol… `GridSearchCV` prueba todas las combinaciones de una lista con validación cruzada y se queda con la mejor. Ajustemos la regresión logística:

In [ ]:
from sklearn.model_selection import GridSearchCV

tubo = Pipeline([("preparar", preparar), ("modelo", LogisticRegression(max_iter=1000))])
rejilla = {"modelo__C": [0.01, 0.03, 0.1, 0.3, 1, 3, 10, 100]}
busqueda = GridSearchCV(tubo, rejilla, cv=grupos)
busqueda.fit(X_ent, y_ent)

for C_, media in zip(rejilla["modelo__C"], busqueda.cv_results_["mean_test_score"]):
    print(f"C = {C_:6}: {media:.1%}")
print("\nMejor:", busqueda.best_params_, f"→ {busqueda.best_score_:.1%}")

(El nombre `modelo__C`, con **dos** guiones bajos, significa «el parámetro `C` del paso llamado `modelo`».)

Con tan pocos datos, varios valores de `C` dan resultados muy parecidos. Eso es normal: no hay que obsesionarse con la décima de porcentaje.

## 9. La evaluación final: abrir la prueba, una sola vez

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, recall_score, precision_score

final = busqueda.best_estimator_          # ya entrenado con todo el entrenamiento
pred = final.predict(X_pru)

print(f"Exactitud en prueba: {np.mean(pred == y_pru):.1%}")
print(f"Línea base en prueba: {max(y_pru.mean(), 1 - y_pru.mean()):.1%}")

# Nos interesan los que NO aprueban (clase 0)
print(f"Sensibilidad para detectar a quienes no aprueban: {recall_score(y_pru, pred, pos_label=0):.1%}")
print(f"Precisión al marcarlos:                           {precision_score(y_pru, pred, pos_label=0):.1%}")

ConfusionMatrixDisplay(confusion_matrix(y_pru, pred), display_labels=["no aprobó", "aprobó"]).plot(cmap="Blues", colorbar=False)
plt.show()

El resultado en la prueba debería parecerse al de la validación cruzada. Si fuera mucho peor, sospecharíamos que algo salió mal (por ejemplo, una fuga de información).

⚠️ **Regla de oro:** después de mirar la prueba ya no se cambia el modelo para «mejorar» ese número. Si lo hicieras, la prueba dejaría de ser honesta (ML 7).

### ¿Y el umbral?

Para las tutorías preferimos detectar a más estudiantes en riesgo (ML 8). Veamos qué pasa con distintos umbrales, usando las probabilidades del modelo:

In [ ]:
p_aprobar = final.predict_proba(X_pru)[:, 1]
print("umbral   marcados   sensibilidad   precisión")
for umbral in [0.3, 0.5, 0.7]:
    marcado = (p_aprobar < umbral).astype(int)        # 1 = en riesgo
    riesgo = (y_pru == 0).astype(int)
    print(f"  {umbral}      {marcado.sum():3}        {recall_score(riesgo, marcado):5.1%}        {precision_score(riesgo, marcado):5.1%}")

💡 Elegir el umbral **mirando la prueba** también sería hacer trampa. Lo correcto sería decidirlo antes, con validación cruzada o con un criterio del colegio («queremos detectar al menos al 90 %»). Aquí lo hacemos solo para ver el efecto.

## 10. Interpretar

In [ ]:
nombres = final.named_steps["preparar"].get_feature_names_out()
coef = final.named_steps["modelo"].coef_[0]
orden = np.argsort(coef)

plt.figure(figsize=(6.5, 4.5))
plt.barh(np.array(nombres)[orden], coef[orden], color=["seagreen" if c > 0 else "crimson" for c in coef[orden]])
plt.axvline(0, color="k", lw=1)
plt.xlabel("coeficiente (columnas normalizadas)")
plt.title("¿Qué empuja hacia aprobar?")
plt.show()

✏️ *¿Qué tres características pesan más? ¿Coincide con lo que viste al explorar? ¿Alguna te sorprende?*

### Limitaciones (¡siempre hay que escribirlas!)

- **Datos simulados:** aquí sabemos que las relaciones fueron inventadas. Con datos reales, un coeficiente **no** prueba que, por ejemplo, desayunar **cause** mejores puntajes (ML 1).
- **Pocos datos:** 240 estudiantes para entrenar. Con más, las conclusiones serían más firmes.
- **¿Justicia?** El modelo usa el colegio. Si se usara para repartir recursos, habría que revisar que no perjudique sistemáticamente a los estudiantes de un colegio. Las decisiones sobre personas siempre deben tener revisión humana.
- **El futuro puede ser distinto:** si cambian la prueba o los estudiantes, el modelo puede dejar de funcionar.

## 11. ⚠️ El error más peligroso: la fuga de información

¿Qué pasaría si, por descuido, dejáramos la columna `puntaje` entre las características?

In [ ]:
con_fuga = datos[numericas + categoricas + ["puntaje"]]
Xf_ent, Xf_pru, yf_ent, yf_pru = train_test_split(con_fuga, y, test_size=0.25, stratify=y, random_state=0)
preparar_fuga = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numericas + ["puntaje"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categoricas),
])
tramposo = Pipeline([("preparar", preparar_fuga), ("modelo", DecisionTreeClassifier(max_depth=2, random_state=0))])
tramposo.fit(Xf_ent, yf_ent)
print(f"Exactitud «con fuga»: {tramposo.score(Xf_pru, yf_pru):.1%}  🤨")

¡Perfecto! Pero es una trampa: el `puntaje` se conoce **después** de la prueba, que es justo lo que queríamos anticipar. Un modelo así no sirve para nada en la vida real.

**Cuando un resultado es demasiado bueno para ser verdad, busca una fuga.** Pregúntate siempre: *¿esta característica la tendría disponible en el momento de predecir?*

## 12. Otros errores comunes

**Error 1: pasarle al `OneHotEncoder` una categoría que no vio.** Por eso usamos `handle_unknown="ignore"`. Sin eso:

In [ ]:
codificador = OneHotEncoder().fit(X_ent[["colegio"]])
codificador.transform(pd.DataFrame({"colegio": ["D"]}))

**Error 2: limpiar con estadísticas de todos los datos.** Rellenar faltantes con el promedio de **todos** los datos (incluida la prueba) es una pequeña fuga. Por eso el `SimpleImputer` va dentro del pipeline.

**Error 3: olvidar la línea base.** «85 % de exactitud» suena bien… salvo que el 85 % de los estudiantes aprueben.

**Error 4: elegir el modelo con la prueba.** Se elige con validación cruzada; la prueba solo confirma.

**Error 5: no documentar la limpieza.** Cada decisión (quitar duplicados, convertir 70 en faltante…) debe quedar escrita, para que otros puedan revisarla.

## 🤔 ¿Qué pasa si…?

Usa la celda de abajo. Antes de ejecutar, **predice** el resultado.

1. …no quitas los duplicados antes de separar entrenamiento y prueba? ¿Por qué podría ser un problema que el mismo estudiante esté en las dos partes?
2. …en lugar de convertir el sueño de 70 horas en faltante, lo dejas así? ¿Cambia mucho la exactitud de la regresión logística con validación cruzada?
3. …quitas `nota_previa` de las características? ¿Cuánto baja la exactitud?
4. …usas `strategy="mean"` en el `SimpleImputer` en lugar de `"median"`?
5. …codificas `colegio` como A = 1, B = 2, C = 3 en lugar de one-hot, y usas un árbol? ¿Y con regresión logística?

In [ ]:
# Prueba aquí tus ideas

## ✍️ Ejercicios del proyecto

**Ejercicio 1 (fácil). El inventario de problemas.** Escribe una tabla (en una celda de texto) con cada problema encontrado en los datos crudos, cuántas filas afectaba y qué decisión tomaste. Usa código para contar cada caso.

**Ejercicio 2 (fácil). Por colegio.** ¿Qué porcentaje aprobó en cada colegio? ¿Y cuál es el promedio de horas de estudio y de redes en cada uno? (Pista: `groupby`, ML 1.)

**Ejercicio 3 (medio). Una característica nueva.** Crea la columna `sueno_suficiente` (1 si duerme al menos 8 horas) o `estudio_por_redes` (horas de estudio divididas entre horas de redes + 1). Agrégala al pipeline. ¿Mejora la validación cruzada?

**Ejercicio 4 (medio). Predecir el puntaje.** Cambia el problema: predice el `puntaje` (un número) en lugar de `aprobo`. Compara con validación cruzada (R²) una regresión lineal, una Ridge (ML 9) y un bosque de regresión (ML 10), usando el mismo `preparar`. ¿Cuál gana?

**Ejercicio 5 (avanzado). Ajustar el bosque.** Usa `GridSearchCV` para el bosque aleatorio con `n_estimators` en [100, 300] y `max_depth` en [3, 5, 8, None] y `max_features` en ["sqrt", 0.5]. ¿Le gana a la regresión logística ajustada? Evalúa en la prueba **solo** el que gane en validación cruzada.

## 🏆 Reto opcional: tu propio proyecto

Elige un conjunto de datos y repite **todas** las etapas, escribiendo el informe de abajo. Algunas opciones:
- `sample_data/california_housing_train.csv` de Colab (predecir el precio de las casas, como en el reto de ML 2);
- `load_breast_cancer()` de scikit-learn (¿tumor benigno o maligno?, con 30 mediciones);
- la encuesta de esta lección, pero prediciendo otra cosa;
- datos que tú mismo recojas con una encuesta en tu colegio (¡pidiendo permiso y sin datos personales!).

### Plantilla de informe

**1. Pregunta:** ¿Qué quiero predecir y para qué?

**2. Datos:** ¿De dónde vienen? ¿Cuántas filas y columnas? ¿Qué problemas tenían y cómo los resolví?

**3. Exploración:** tres observaciones con sus gráficas.

**4. Modelos:** línea base y candidatos, con sus resultados de validación cruzada.

**5. Resultado final:** el modelo elegido y su desempeño en la prueba (una sola vez).

**6. Interpretación:** qué características importan más.

**7. Limitaciones y riesgos:** qué no sabemos, a quién podría perjudicar, qué haría con más tiempo o datos.

## 🎬 Cierre

En esta sección recorriste el camino completo del machine learning clásico:
- **Datos** con pandas (ML 1), limpieza y exploración.
- **Modelos:** regresión lineal (ML 2), descenso de gradiente (ML 3), k vecinos (ML 4), árboles (ML 5), k-means (ML 6), regresión logística (ML 8), regresión polinomial con regularización (ML 9), bosques y ensambles (ML 10) y PCA (ML 11).
- **Evaluación honesta:** entrenamiento y prueba, validación cruzada, matriz de confusión (ML 7) y, hoy, líneas base, pipelines completos y fugas de información.

Y lo más importante: un buen proyecto no es el que tiene el número más alto, sino el que responde una pregunta útil, con un método **honesto** y unas **limitaciones** bien explicadas.

**Para pensar:** todos estos modelos trabajan con tablas de características que **nosotros** elegimos. ¿Y si los datos son imágenes, sonidos o textos, donde no es obvio qué características usar? En la **Sección 4, Redes neuronales**, empezaremos por la neurona artificial más simple, el **perceptrón** (RN 1), y llegaremos a redes que aprenden sus propias características con PyTorch. 🧠🚀